# Demo 2: Gemini API + MySQL Analysis
## REY x Mikroskil — Praktisi Mengajar: Basis Data

### Cara Penggunaan:
1. **Setup environment**: `make setup`
2. **Set API key**: `cp .env.example .env` lalu edit `.env`
3. **Pastikan MySQL running**: `make up`
4. **Jalankan cell satu per satu** dari atas ke bawah

Setiap cell adalah satu langkah. Output muncul langsung di bawah cell.

> **Google Colab users**: Upload notebook ini ke Colab, lalu jalankan cell secara berurutan. Colab sudah termasuk Python runtime — tinggal install dependencies dan set API key secara manual.

## Setup: Imports & Configuration
Jalankan cell ini terlebih dahulu.

In [1]:
from dotenv import load_dotenv
import os

# Load API key from .env file (if exists)
load_dotenv()

import mysql.connector
import json
import re
from google import genai

# Konfigurasi MySQL (sesuaikan dengan docker-compose)
MYSQL_CONFIG = {
    "host": os.environ.get("MYSQL_HOST", "localhost"),
    "port": int(os.environ.get("MYSQL_PORT", "3306")),
    "user": os.environ.get("MYSQL_USER", "root"),
    "password": os.environ.get("MYSQL_PASSWORD", "password"),
    "database": os.environ.get("MYSQL_DATABASE", "db_rey_mikroskil"),
}

# Konfigurasi Gemini
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
GEMINI_MODEL = "gemini-2.0-flash"

if not GEMINI_API_KEY:
    raise SystemExit(
        "GEMINI_API_KEY tidak ditemukan di .env file!\n"
        "  1. Copy .env.example -> .env\n"
        "  2. Edit .env dan paste API key kamu\n"
    )

client = genai.Client(api_key=GEMINI_API_KEY)
print("Setup complete! Imports OK, Gemini client initialized.")

Setup complete! Imports OK, Gemini client initialized.


## Function Definitions
Semua fungsi helper didefinisikan di sini. Jalankan cell ini sebelum menjalankan step-step di bawahnya.

In [2]:
def get_connection():
    """Buka koneksi MySQL baru."""
    return mysql.connector.connect(**MYSQL_CONFIG)


def get_schema_info():
    """Ambil schema semua tabel sebagai teks terstruktur."""
    conn = get_connection()
    cursor = conn.cursor(dictionary=True)
    cursor.execute("""
        SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE, COLUMN_TYPE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = %(db)s
        ORDER BY TABLE_NAME, ORDINAL_POSITION
    """, {"db": MYSQL_CONFIG["database"]})
    
    schema = {}
    for row in cursor.fetchall():
        tname = row["TABLE_NAME"]
        if tname not in schema:
            schema[tname] = []
        schema[tname].append(f"    {row['COLUMN_NAME']}  {row['COLUMN_TYPE']}")
    cursor.close()
    conn.close()
    
    lines = []
    for tname, cols in schema.items():
        lines.append(f"Table: {tname}")
        lines.extend(cols)
        lines.append("")
    return "\n".join(lines), schema


def ask_gemini_sql(question, schema_text):
    """Minta Gemini generate SQL query dari schema + pertanyaan bisnis."""
    prompt = f"""Kamu adalah seorang Analytics Engineer expert di MySQL.

Berikut adalah schema database yang tersedia:
{schema_text}

Pertanyaan bisnis: {question}

Tulis SATU SQL query untuk menjawab pertanyaan tersebut.
ATURAN:
- HANYA tulis SQL query saja, TANPA penjelasan atau markdown
- Gunakan nama tabel dan kolom PERSIS sesuai schema di atas
- Jika query membutuhkan aggregation, gunakan GROUP BY yang tepat
"""
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    sql = response.text.strip()
    # Bersihkan markdown code blocks jika Gemini menambahkannya
    sql = re.sub(r'^```(?:sql)?\s*', '', sql, flags=re.MULTILINE | re.IGNORECASE)
    sql = re.sub(r'\s*```$', '', sql, flags=re.MULTILINE)
    # Strip everything after first semicolon (Gemini sometimes adds multiple queries)
    sql = sql.split(';')[0].strip()
    # Fix missing space: 'ASSELECT' -> 'AS SELECT'
    sql = re.sub(r'(AS)(SELECT|FROM|WHERE)', r'\1 \2', sql, flags=re.IGNORECASE)
    return sql.strip()


def execute_query(sql):
    """Eksekusi SQL query, return (columns, rows)."""
    conn = get_connection()
    cursor = conn.cursor(dictionary=True)
    try:
        cursor.execute(sql)
        results = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        return columns, results
    except mysql.connector.Error as e:
        print(f"  ERROR: {e}")
        return [], []
    finally:
        cursor.close()
        conn.close()


def ask_gemini_insights(question, sql, columns, results):
    """Minta Gemini menganalisis hasil query dan memberikan insight bisnis."""
    results_str = json.dumps(results, indent=2, default=str)
    prompt = f"""Seorang Analytics Engineer menjalankan SQL query untuk menjawab pertanyaan bisnis.

Pertanyaan: {question}
SQL Query: {sql}
Hasil Query (JSON):
{results_str}

Jelaskan insight dari hasil query ini dalam bahasa Indonesia.
Berikan:
1. Ringkasan temuan utama (2-3 kalimat)
2. Kesimpulan bisnis yang actionable
3. Jika ada anomaly atau pola menarik, sebutkan
"""
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    return response.text


print("All functions defined! Ready to run steps.")

All functions defined! Ready to run steps.


## Step 1: Test Koneksi MySQL
Verifikasi bahwa Python bisa terhubung ke database MySQL di Docker.

In [3]:
print("=" * 60)
print("STEP 1: Testing MySQL Connection")
print("=" * 60)
try:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SHOW TABLES")
    tables = [row[0] for row in cursor.fetchall()]
    print(f"  Connected! Found {len(tables)} tables:")
    for t in tables:
        print(f"    - {t}")
    cursor.close()
    conn.close()
    print("\n  Koneksi berhasil! Lanjut ke Step 2.")
except mysql.connector.Error as e:
    print(f"  Koneksi GAGAL: {e}")
    print("  Pastikan Docker MySQL sudah running:")
    print("    cd Topik\ 2 && docker compose up -d")

STEP 1: Testing MySQL Connection
  Connected! Found 8 tables:
    - health_diaries
    - reyfit_logs
    - subscriptions
    - user_engagement_scores
    - user_logins
    - users
    - view_raw_activities
    - view_user_cohort

  Koneksi berhasil! Lanjut ke Step 2.


## Step 2: Ambil Schema Database
Kita ambil informasi kolom dan tipe data dari semua tabel. Ini akan jadi konteks untuk Gemini.

In [4]:
schema_text, schema_dict = get_schema_info()
print("=" * 60)
print("STEP 2: Schema Database")
print("=" * 60)
print(schema_text)
print(f"\nTotal: {len(schema_dict)} tables loaded as context for Gemini.")

STEP 2: Schema Database
Table: health_diaries
    diary_id  int
    user_id  int
    active_month  date
    count_diaries  int

Table: reyfit_logs
    log_id  int
    user_id  int
    active_month  date
    step_counts  int
    total_hydration_ml  int

Table: subscriptions
    subscription_id  int
    user_id  int
    start_date  date
    end_date  date
    status  varchar(20)

Table: user_engagement_scores
    user_id  int
    total_activity  int
    z_score  float
    normalized_score  float
    segment  varchar(50)
    updated_at  timestamp

Table: user_logins
    login_id  int
    user_id  int
    platform  varchar(50)
    last_login_at  datetime

Table: users
    user_id  int
    first_activity  date
    last_activity  date

Table: view_raw_activities
    user_id  int
    raw_activity  bigint

Table: view_user_cohort
    user_id  int
    start_date  date
    end_date  date
    cohort_month  varchar(10)
    sub_age_months  bigint


Total: 8 tables loaded as context for Gemini.


## Step 3: Gemini Generate SQL
Kita beri Gemini schema database + pertanyaan bisnis, dan minta dia buatkan SQL query.

In [5]:
question_1 = "Berapa jumlah user di setiap cohort bulan berdasarkan start_date mereka? Kelompokkan berdasarkan tahun-bulan dan urutkan dari yang paling awal."

print("=" * 60)
print("STEP 3: Gemini Generate SQL")
print("=" * 60)
print(f"\n  Pertanyaan: {question_1}")
sql_1 = ask_gemini_sql(question_1, schema_text)
print(f"\n  Gemini-generated SQL:")
print(f"  {sql_1}")

STEP 3: Gemini Generate SQL

  Pertanyaan: Berapa jumlah user di setiap cohort bulan berdasarkan start_date mereka? Kelompokkan berdasarkan tahun-bulan dan urutkan dari yang paling awal.

  Gemini-generated SQL:
  SELECT
    cohort_month,
    COUNT(DISTINCT user_id) AS total_users
FROM
    view_user_cohort
GROUP BY
    cohort_month
ORDER BY
    cohort_month


## Step 4: Eksekusi Query dari Gemini
Jalankan SQL yang dibuat Gemini terhadap database MySQL.

In [6]:
print("=" * 60)
print("STEP 4: Executing Gemini's SQL Query")
print("=" * 60)
columns_1, results_1 = execute_query(sql_1)
if results_1:
    print(f"\n  Query berhasil! {len(results_1)} rows returned:")
    for row in results_1:
        print(f"    {row}")
    print(f"\n  Lanjut ke Step 5 untuk analisis dari Gemini.")
else:
    print("  Query gagal atau tidak ada hasil.")
    print("  Cek apakah SQL yang dihasilkan Gemini valid.")

STEP 4: Executing Gemini's SQL Query

  Query berhasil! 18 rows returned:
    {'cohort_month': '2024-01-01', 'total_users': 78}
    {'cohort_month': '2024-02-01', 'total_users': 80}
    {'cohort_month': '2024-03-01', 'total_users': 10}
    {'cohort_month': '2024-04-01', 'total_users': 548}
    {'cohort_month': '2024-05-01', 'total_users': 237}
    {'cohort_month': '2024-06-01', 'total_users': 124}
    {'cohort_month': '2024-07-01', 'total_users': 895}
    {'cohort_month': '2024-08-01', 'total_users': 52}
    {'cohort_month': '2024-09-01', 'total_users': 1517}
    {'cohort_month': '2024-10-01', 'total_users': 83}
    {'cohort_month': '2024-11-01', 'total_users': 68}
    {'cohort_month': '2024-12-01', 'total_users': 112}
    {'cohort_month': '2025-01-01', 'total_users': 790}
    {'cohort_month': '2025-02-01', 'total_users': 303}
    {'cohort_month': '2025-03-01', 'total_users': 113}
    {'cohort_month': '2025-04-01', 'total_users': 359}
    {'cohort_month': '2025-05-01', 'total_users': 6

## Step 5: Gemini Analisis Hasil Query
Kirim hasil query balik ke Gemini untuk mendapatkan insight dalam bahasa Indonesia.

In [7]:
print("=" * 60)
print("STEP 5: Gemini Analyzes Results")
print("=" * 60)
insights_1 = ask_gemini_insights(question_1, sql_1, columns_1, results_1)
print(f"\n  Gemini Insight:")
print(f"{insights_1}")

STEP 5: Gemini Analyzes Results

  Gemini Insight:
Berikut adalah analisis dari hasil query tersebut:

**1. Ringkasan Temuan Utama:**

*   Jumlah user yang bergabung (cohort size) bervariasi signifikan dari bulan ke bulan.
*   Ada beberapa bulan dengan jumlah user yang sangat tinggi (September 2024, Juli 2024, Januari 2025), sementara bulan lain memiliki jumlah user yang jauh lebih rendah.
*   Cohort di akhir periode (Mei dan Juni 2025) menunjukkan jumlah user yang sangat kecil.

**2. Kesimpulan Bisnis yang Actionable:**

*   **Investigasi Peningkatan Tajam:**  Tim marketing dan product perlu menyelidiki faktor-faktor yang menyebabkan lonjakan jumlah user di bulan September 2024, Juli 2024, dan Januari 2025. Apakah ada kampanye pemasaran khusus, fitur baru yang populer, atau faktor eksternal lainnya? Jika berhasil diidentifikasi, strategi serupa dapat diimplementasikan untuk meningkatkan akuisisi user di bulan-bulan lainnya.
*   **Identifikasi Masalah Akuisisi User:** Jumlah user yang 

## Step 6: Full Pipeline — 3 Skenario Demo
Jalankan semua skenario secara otomatis. Ini menunjukkan kekuatan penuh dari pipeline AI + MySQL.

In [8]:
print("=" * 60)
print("DEMO 2: Full Pipeline — 3 Skenario Analisis")
print("=" * 60)

# Ambil schema untuk konteks Gemini
schema_text, schema_dict = get_schema_info()
print(f"\n  Schema loaded: {len(schema_dict)} tables found")

# 3 Skenario Analisis
scenarios = [
    {
        "title": "Skenario 1: Cohort Retention",
        "question": "Berapa jumlah user di setiap cohort bulan berdasarkan start_date mereka? Kelompokkan berdasarkan tahun-bulan dan urutkan dari yang paling awal.",
    },
    {
        "title": "Skenario 2: User Engagement Breakdown",
        "question": "Untuk setiap user_id, hitung: (a) jumlah record di user_logins, (b) jumlah record di reyfit_logs, (c) jumlah record di health_diaries. Tampilkan top 10 user diurutkan dari total aktivitas tertinggi.",
    },
    {
        "title": "Skenario 3: Subscription Duration Analysis",
        "question": "Hitung berapa hari setiap user sudah berlangganan (dari start_date sampai COALESCE(end_date, CURDATE())). Urutkan dari yang paling lama. Batasi top 10. Tampilkan user_id, durasi hari, status, dan last_activity dari view users.",
    },
]

for i, scenario in enumerate(scenarios, 1):
    print(f"\n{'=' * 60}")
    print(f"  {scenario['title']}")
    print(f"{'=' * 60}")
    print(f"\n  Pertanyaan: {scenario['question']}")

    # A. Gemini generates SQL
    print("\n  [A] Gemini generating SQL...")
    sql = ask_gemini_sql(scenario["question"], schema_text)
    print(f"  SQL: {sql}")

    # B. Execute query
    print("\n  [B] Executing query...")
    cols, res = execute_query(sql)
    if res:
        print(f"  Result: {len(res)} rows")
        for row in res:
            print(f"    {row}")

        # C. Gemini analyzes
        print("\n  [C] Gemini analyzing results...")
        insight = ask_gemini_insights(scenario["question"], sql, cols, res)
        print(f"\n  Insight:\n{insight}")
    else:
        print("  Query gagal atau tidak ada hasil.")

print("\n" + "=" * 60)
print("Demo selesai!")
print("=" * 60)

DEMO 2: Full Pipeline — 3 Skenario Analisis

  Schema loaded: 8 tables found

  Skenario 1: Cohort Retention

  Pertanyaan: Berapa jumlah user di setiap cohort bulan berdasarkan start_date mereka? Kelompokkan berdasarkan tahun-bulan dan urutkan dari yang paling awal.

  [A] Gemini generating SQL...
  SQL: SELECT
    cohort_month,
    COUNT(DISTINCT user_id) AS total_users
FROM
    view_user_cohort
GROUP BY
    cohort_month
ORDER BY
    cohort_month

  [B] Executing query...
  Result: 18 rows
    {'cohort_month': '2024-01-01', 'total_users': 78}
    {'cohort_month': '2024-02-01', 'total_users': 80}
    {'cohort_month': '2024-03-01', 'total_users': 10}
    {'cohort_month': '2024-04-01', 'total_users': 548}
    {'cohort_month': '2024-05-01', 'total_users': 237}
    {'cohort_month': '2024-06-01', 'total_users': 124}
    {'cohort_month': '2024-07-01', 'total_users': 895}
    {'cohort_month': '2024-08-01', 'total_users': 52}
    {'cohort_month': '2024-09-01', 'total_users': 1517}
    {'cohor